In [3]:
# use alpha 2023 to calculate initial cr 2023
import pandas as pd
import numpy as np
from scipy.special import gamma, gammaincc
pd.set_option('display.float_format', '{:.6f}'.format)

# Gamma parameters
gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584
def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z) 
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)
    return num / den

# =============================================================================
# 0. Read data
# =============================================================================
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv')
new_df['csemCur'] = new_df['csemCur'].astype(int)
Age_2023_fixed = new_df.groupby('csemCur')['Age 2023'].first().tolist()

# =============================================================================
# 1. Parameters
# =============================================================================
T, base_year, Y = 17, 2024, 365
mu_coper, mu_mant = 1.6, 4.2
C_MANT = C_COPER = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T)]
C_leak_t = [c * 25200 for c in c_factors]
nb_coper, nb_mant = 7, 36

# =============================================================================
# 2. 2023 baseline
# =============================================================================
df_alpha_2023 = new_df.groupby("csemCur")[["alpha", "Pression Nominale Relative", "CR_threshold", "Volume"]].last().reset_index()
df_alpha_2023.columns = ["csemCur", "alpha_2023", "Pression Nominale Relative", "CR_threshold", "Volume"]
print("2023 baseline:")
print(df_alpha_2023.round(4))

# =============================================================================
# 3. Initialize states
# =============================================================================
csem_states = {}
for i, row in df_alpha_2023.iterrows():
    csemCur = int(row["csemCur"])
    alpha, delta_p = row["alpha_2023"], row["Pression Nominale Relative"] - row["CR_threshold"]
    cr_init = Y * alpha / delta_p if delta_p > 0 else 0
    
    csem_states[csemCur] = {
        'cr_current': cr_init,
        'age_2023_fixed': Age_2023_fixed[i],  # ✅ FIXED
        'current_age': Age_2023_fixed[i] + 1,  # 2024开始
        'alpha': alpha, 'pression_nominale': row["Pression Nominale Relative"],
        'cr_threshold': row["CR_threshold"], 'volume_csem': row["Volume"],
        'csem_K': 146 * row["Volume"] / (8.314 * 293.15 * 10**3),
        'pending_task': None  # ✅ 添加这行
    }
    print(f"CSEM {csemCur}: age_2023_fixed={Age_2023_fixed[i]:.1f}, cr_init={cr_init:.3f}, alpha={alpha:.6f}, delta_p={delta_p:.3f}")

# # =============================================================================
# # 4. Simulation
# # =============================================================================
# rows = []
# for t in range(T):
#     year = base_year + t
#     year_offset = year - 2023  # 2024=1, 2025=2...
    
#     # Find triggered
#     triggered_csem = []
#     for csemCur, state in csem_states.items():
#         cr, delta_p = state['cr_current'], state['pression_nominale'] - state['cr_threshold']
#         leak = cr * delta_p * state['csem_K']
#         interval = 365 / cr if cr > 0 else np.inf
#         if leak >= 5 or interval <= 60:
#             triggered_csem.append({'csemCur': csemCur, 'leak': leak, 'state': state})
    
#     triggered_csem.sort(key=lambda x: x['leak'], reverse=True)
#     task_assignments = {}
#     for i, info in enumerate(triggered_csem):
#         csemCur = info['csemCur']
#         if i < nb_mant: task_assignments[csemCur] = 'mant'
#         elif i < nb_mant + nb_coper: task_assignments[csemCur] = 'coper'
    
#     # Process all
#     for csemCur, state in csem_states.items():
#         cr, delta_p = state['cr_current'], state['pression_nominale'] - state['cr_threshold']
#         leak = cr * delta_p * state['csem_K']
#         interval = 365 / cr if cr > 0 else np.inf
#         triggered = (leak >= 5) or (interval <= 60)
        
#         leak_cost = leak * C_leak_t[t]
#         task_type = task_assignments.get(csemCur)
#         action_cost = { 'mant': C_MANT, 'coper': C_COPER }.get(task_type, 0)
        
#         rows.append(dict(csemCur=csemCur, year=year, cr=cr, leak=leak, interval=interval,
#                         triggered=triggered, task_type=task_type,
#                         leak_cost=leak_cost, action_cost=action_cost, total_cost=leak_cost+action_cost))
        
#         # Update for next year
#         next_offset = year_offset + 1
#         next_age = state['age_2023_fixed'] + next_offset
        
#         if task_type == 'mant':
#             state['cr_current'] /= mu_mant
#         elif task_type == 'coper':
#             state['cr_current'] /= mu_coper
#         else:
#             state['cr_current'] += compute_gamma_h(next_age, gamma_alpha, gamma_beta)
#         state['current_age'] = next_age




# 新替换的恶simulation部分，解决任务没有出现在下一年而是当年的问题
# 在初始化csem_states时加 'pending_task': None

rows = []
for t in range(T):
    year = base_year + t
    year_offset = year - 2023 
    
    # 应用上一年任务（参考代码的triggered_previous_year逻辑）
    applied_tasks = {}
    for csemCur, state in csem_states.items():
        if state['pending_task']:
            task = state['pending_task']
            if task == 'mant': state['cr_current'] /= mu_mant
            elif task == 'coper': state['cr_current'] /= mu_coper
            applied_tasks[csemCur] = task
            state['pending_task'] = None
    
    # 决策（同原）
    triggered_csem = []
    for csemCur, state in csem_states.items():
        cr, delta_p = state['cr_current'], state['pression_nominale'] - state['cr_threshold']
        leak = cr * delta_p * (10**5) * state['csem_K']
        interval = 365 / cr if cr > 0 else np.inf
        if leak >= 5 or interval <= 60:
            triggered_csem.append({'csemCur': csemCur, 'leak': leak, 'state': state})
    
    triggered_csem.sort(key=lambda x: x['leak'], reverse=True)
    task_assignments = {}
    for i, info in enumerate(triggered_csem):
        csemCur = info['csemCur']
        if i < nb_mant: task_assignments[csemCur] = 'mant'
        elif i < nb_mant + nb_coper: task_assignments[csemCur] = 'coper'
    
    # 存入下一年pending
    for csemCur in task_assignments:
        csem_states[csemCur]['pending_task'] = task_assignments[csemCur]
    
    # 记录rows（task_type=applied this year）
    for csemCur, state in csem_states.items():
        cr, delta_p = state['cr_current'], state['pression_nominale'] - state['cr_threshold']
        leak = cr * delta_p * (10**5) * state['csem_K']
        interval = 365 / cr if cr > 0 else np.inf
        triggered = (leak >= 5) or (interval <= 60)
        task_type = applied_tasks.get(csemCur)
        
        leak_cost = leak * C_leak_t[t]
        action_cost = {'mant': C_MANT, 'coper': C_COPER}.get(task_type, 0)
        
        rows.append(dict(csemCur=csemCur, year=year, cr=cr, leak=leak, interval=interval,
                        triggered=triggered, task_type=task_type,
                        leak_cost=leak_cost, action_cost=action_cost, total_cost=leak_cost+action_cost))
    
    # 年龄更新
    next_offset = year_offset + 1
    for state in csem_states.values():
        next_age = state['age_2023_fixed'] + next_offset
        state['current_age'] = next_age
        if state['pending_task'] is None:  # 无任务才退化
            state['cr_current'] += compute_gamma_h(next_age, gamma_alpha, gamma_beta)




# =============================================================================
# 5. Results (完全修复merge逻辑)
# =============================================================================
sim_df = pd.DataFrame(rows)
sim_df['csemCur'] = sim_df['csemCur'].astype(int)

# ✅ 创建criteria列
sim_df['criterion_environment'] = sim_df['leak'] >= 5
sim_df['criterion_exploit'] = sim_df['interval'] <= 60

# ✅ Summary DataFrames (所有列名统一为csemCur开头)
cost_summary = sim_df.groupby("csemCur")["total_cost"].sum().round(0).reset_index()
cost_summary.columns = ["csemCur", "total_T years_cost"]

leak_summary = sim_df.groupby("csemCur")["leak"].sum().round(3).reset_index()
leak_summary.columns = ["csemCur", "total_leak_kg"]

trigger_exploit = sim_df.groupby("csemCur")["criterion_exploit"].sum().astype(int).reset_index()
trigger_exploit.columns = ["csemCur", "trigger_exploit_times"]

trigger_env = sim_df.groupby("csemCur")["criterion_environment"].sum().astype(int).reset_index()
trigger_env.columns = ["csemCur", "trigger_env_times"]

total_years_df = sim_df.groupby("csemCur").size().reset_index(name='total_years')

mant_count = (sim_df[sim_df['task_type'] == 'mant']
              .groupby('csemCur').size()
              .reindex(cost_summary['csemCur'], fill_value=0)
              .reset_index(name='mant_times'))

coper_count = (sim_df[sim_df['task_type'] == 'coper']
               .groupby('csemCur').size()
               .reindex(cost_summary['csemCur'], fill_value=0)
               .reset_index(name='coper_times'))

# ✅ 顺序merge，确保所有DataFrame都有csemCur列
summary_df = cost_summary
summary_df = summary_df.merge(leak_summary, on='csemCur', how='outer')
summary_df = summary_df.merge(trigger_exploit, on='csemCur', how='outer')
summary_df = summary_df.merge(trigger_env, on='csemCur', how='outer')
summary_df = summary_df.merge(total_years_df, on='csemCur', how='outer')
summary_df = summary_df.merge(mant_count, on='csemCur', how='outer')
summary_df = summary_df.merge(coper_count, on='csemCur', how='outer')

# ✅ 填充NaN并计算比率
summary_df = summary_df.fillna(0)
summary_df['total_years'] = summary_df['total_years'].astype(int)
summary_df['trigger_exploit_rate'] = (summary_df['trigger_exploit_times'] / summary_df['total_years']).round(3)
summary_df['trigger_env_rate'] = (summary_df['trigger_env_times'] / summary_df['total_years']).round(3)
summary_df['total_trigger_rate'] = ((summary_df['trigger_exploit_times'] + summary_df['trigger_env_times']) / summary_df['total_years']).round(3)

print("\n📊 Summary:")
print(summary_df.round(3))
print("\n📊 Top 10 by total_leak_kg:")
print(summary_df.sort_values('total_leak_kg', ascending=False).head(10).round(3))

# Top 10详细轨迹
print("\n" + "="*80)
print("🔥 Top 10 CSEM results")
print("="*80)
top_csem = summary_df.nlargest(10, 'total_leak_kg')['csemCur'].tolist()
for csem_id in top_csem:
    print(f"\n📋 CSEM {csem_id}:")
    detail = sim_df[sim_df['csemCur'] == csem_id][['year', 'cr', 'leak', 'interval', 'triggered', 'task_type', 'total_cost']].round({'cr':4, 'leak':1})
    print(detail.to_string(index=False))

# 保存
filename = "simulation_cr_fixed_age_final"
sim_df.to_csv(f"{filename}.csv", index=False)
summary_df.to_csv(f"{filename}_summary.csv", index=False)
print(f"\n✅ Saved {filename}.csv & {filename}_summary.csv")


2023 baseline:
        csemCur  alpha_2023  Pression Nominale Relative  CR_threshold   Volume
0    5000252447    0.000300                    3.500000      3.199000 6.317600
1    5000252448    0.000500                    3.500000      3.199000 6.317600
2    5000252449    0.000100                    3.500000      3.199000 6.302500
3    5000252450    0.000900                    3.500000      3.199000 9.806700
4    5000252524    0.000400                    3.500000      3.199000 0.443900
..          ...         ...                         ...           ...      ...
285  5001342228    0.001000                    3.500000      3.199000 1.907300
286  5001342229    0.000300                    3.500000      3.199000 1.907300
287  5002343705    0.000300                    3.500000      3.199000 0.952800
288  5002343706    0.000100                    3.500000      3.199000 0.952800
289  5002343707    0.000200                    3.500000      3.199000 0.952800

[290 rows x 5 columns]
CSEM 50002524

In [7]:
# use directly cr 2023 filtered in the data
import pandas as pd
import numpy as np
from scipy.special import gamma, gammaincc
pd.set_option('display.float_format', '{:.6f}'.format)

# Gamma parameters
gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584
def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z) 
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)
    return num / den

# =============================================================================
# 0. Read data
# =============================================================================
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv')
new_df['csemCur'] = new_df['csemCur'].astype(int)
Age_2023_fixed = new_df.groupby('csemCur')['Age 2023'].first().tolist()

# =============================================================================
# 1. Parameters
# =============================================================================
T, base_year, Y = 17, 2024, 365
mu_coper, mu_mant = 1.6, 4.2
C_MANT = 48000
C_COPER = 15600
c_factors = [0.0952 + 0.004 * t for t in range(T)]
C_leak_t = [c * 25200 for c in c_factors]
nb_coper, nb_mant = 3, 14


# =============================================================================
# 2. 2023 baseline
# =============================================================================
df_alpha_2023 = new_df.groupby("csemCur")[["alpha", "Pression Nominale Relative", "CR_threshold", "Volume", "cr"]].last().reset_index()
df_alpha_2023.columns = ["csemCur", "alpha_2023", "Pression Nominale Relative", "CR_threshold", "Volume", "cr"]
print("2023 baseline:")
print(df_alpha_2023.round(4))

# =============================================================================
# 3. Initialize states
# =============================================================================
csem_states = {}
for i, row in df_alpha_2023.iterrows():
    csemCur = int(row["csemCur"])
    alpha, delta_p = row["alpha_2023"], row["Pression Nominale Relative"] - row["CR_threshold"]
    cr_init = row["cr"] if row["cr"] > 0 else 0
    
    csem_states[csemCur] = {
        'cr_current': cr_init,
        'age_2023_fixed': Age_2023_fixed[i],  # ✅ FIXED
        'current_age': Age_2023_fixed[i] + 1,  # 2024开始
        'alpha': alpha, 'pression_nominale': row["Pression Nominale Relative"],
        'cr_threshold': row["CR_threshold"], 'volume_csem': row["Volume"],
        'csem_K': 146 * row["Volume"] / (8.314 * 293.15 * 10**3)
    }
    print(f"CSEM {csemCur}: age_2023_fixed={Age_2023_fixed[i]:.1f}, cr_init={cr_init:.3f}, alpha={alpha:.6f}, delta_p={delta_p:.3f}")

# =============================================================================
# 4. Simulation
# =============================================================================
rows = []
for t in range(T):
    year = base_year + t
    year_offset = year - 2023  # 2024=1, 2025=2...
    
    # Find triggered
    triggered_csem = []
    for csemCur, state in csem_states.items():
        cr, delta_p = state['cr_current'], state['pression_nominale'] - state['cr_threshold']
        leak = cr * delta_p * (10**5) * state['csem_K']
        interval = 365 / cr if cr > 0 else np.inf
        if leak >= 5 or interval <= 60:
            triggered_csem.append({'csemCur': csemCur, 'leak': leak, 'state': state})
    
    triggered_csem.sort(key=lambda x: x['leak'], reverse=True)
    task_assignments = {}
    for i, info in enumerate(triggered_csem):
        csemCur = info['csemCur']
        if i < nb_mant: task_assignments[csemCur] = 'mant'
        elif i < nb_mant + nb_coper: task_assignments[csemCur] = 'coper'
    
    # Process all
    for csemCur, state in csem_states.items():
        cr, delta_p = state['cr_current'], state['pression_nominale'] - state['cr_threshold']
        leak = cr * delta_p * (10**5) * state['csem_K']
        interval = 365 / cr if cr > 0 else np.inf
        triggered = (leak >= 5) or (interval <= 60)
        
        leak_cost = leak * C_leak_t[t]
        task_type = task_assignments.get(csemCur)
        action_cost = { 'mant': C_MANT, 'coper': C_COPER }.get(task_type, 0)
        
        rows.append(dict(csemCur=csemCur, year=year, cr=cr, leak=leak, interval=interval,
                        triggered=triggered, task_type=task_type,
                        leak_cost=leak_cost, action_cost=action_cost, total_cost=leak_cost+action_cost))
        
        # Update for next year
        next_offset = year_offset + 1
        next_age = state['age_2023_fixed'] + next_offset
        
        if task_type == 'mant':
            state['cr_current'] /= mu_mant
        elif task_type == 'coper':
            state['cr_current'] /= mu_coper
        else:
            state['cr_current'] += compute_gamma_h(next_age, gamma_alpha, gamma_beta)
        state['current_age'] = next_age

# =============================================================================
# 5. Results (完全修复merge逻辑)
# =============================================================================
sim_df = pd.DataFrame(rows)
sim_df['csemCur'] = sim_df['csemCur'].astype(int)

# ✅ 创建criteria列
sim_df['criterion_environment'] = sim_df['leak'] >= 5
sim_df['criterion_exploit'] = sim_df['interval'] <= 60

# ✅ Summary DataFrames (所有列名统一为csemCur开头)
cost_summary = sim_df.groupby("csemCur")["total_cost"].sum().round(0).reset_index()
cost_summary.columns = ["csemCur", "total_T years_cost"]

leak_summary = sim_df.groupby("csemCur")["leak"].sum().round(3).reset_index()
leak_summary.columns = ["csemCur", "total_leak_kg"]

trigger_exploit = sim_df.groupby("csemCur")["criterion_exploit"].sum().astype(int).reset_index()
trigger_exploit.columns = ["csemCur", "trigger_exploit_times"]

trigger_env = sim_df.groupby("csemCur")["criterion_environment"].sum().astype(int).reset_index()
trigger_env.columns = ["csemCur", "trigger_env_times"]

total_years_df = sim_df.groupby("csemCur").size().reset_index(name='total_years')

mant_count = (sim_df[sim_df['task_type'] == 'mant']
              .groupby('csemCur').size()
              .reindex(cost_summary['csemCur'], fill_value=0)
              .reset_index(name='mant_times'))

coper_count = (sim_df[sim_df['task_type'] == 'coper']
               .groupby('csemCur').size()
               .reindex(cost_summary['csemCur'], fill_value=0)
               .reset_index(name='coper_times'))

# ✅ 顺序merge，确保所有DataFrame都有csemCur列
summary_df = cost_summary
summary_df = summary_df.merge(leak_summary, on='csemCur', how='outer')
summary_df = summary_df.merge(trigger_exploit, on='csemCur', how='outer')
summary_df = summary_df.merge(trigger_env, on='csemCur', how='outer')
summary_df = summary_df.merge(total_years_df, on='csemCur', how='outer')
summary_df = summary_df.merge(mant_count, on='csemCur', how='outer')
summary_df = summary_df.merge(coper_count, on='csemCur', how='outer')

# ✅ 填充NaN并计算比率
summary_df = summary_df.fillna(0)
summary_df['total_years'] = summary_df['total_years'].astype(int)
summary_df['trigger_exploit_rate'] = (summary_df['trigger_exploit_times'] / summary_df['total_years']).round(3)
summary_df['trigger_env_rate'] = (summary_df['trigger_env_times'] / summary_df['total_years']).round(3)
summary_df['total_trigger_rate'] = ((summary_df['trigger_exploit_times'] + summary_df['trigger_env_times']) / summary_df['total_years']).round(3)

print("\n📊 Summary:")
print(summary_df.round(3))
print("\n📊 Top 10 by total_leak_kg:")
print(summary_df.sort_values('total_leak_kg', ascending=False).head(10).round(3))

# Top 10详细轨迹
print("\n" + "="*80)
print("🔥 Every CSEM's Detailed Results (Top 10 by Total Leak)")
print("="*80)
top_csem = summary_df.nlargest(10, 'total_leak_kg')['csemCur'].tolist()
for csem_id in top_csem:
    print(f"\n📋 CSEM {csem_id}:")
    detail = sim_df[sim_df['csemCur'] == csem_id][['year', 'cr', 'leak', 'interval', 'triggered', 'task_type', 'total_cost']].round({'cr':4, 'leak':1})
    print(detail.to_string(index=False))


print('Total leak (kg) across all csemCur over entire horizon:', sim_df['leak'].sum().round(3))
print('Total cost across all csemCur over entire horizon:', sim_df['total_cost'].sum().round(0))
print('Total action cost across all csemCur over entire horizon:', sim_df['action_cost'].sum().round(0))
print('Total leak cost across all csemCur over entire horizon:', sim_df['leak_cost'].sum().round(0))
print('Total MANT actions:', sim_df[sim_df['task_type'] == 'mant'].shape[0])
print('Total COPER actions:', sim_df[sim_df['task_type'] == 'coper'].shape[0])

# 保存
# filename = "simulation_cr_fixed_age_final"
sim_df.to_csv(f"{filename}.csv", index=False)
summary_df.to_csv(f"{filename}_summary.csv", index=False)
print(f"\n✅ Saved {filename}.csv & {filename}_summary.csv")


2023 baseline:
        csemCur  alpha_2023  Pression Nominale Relative  CR_threshold  \
0    5000252447    0.000300                    3.500000      3.199000   
1    5000252448    0.000500                    3.500000      3.199000   
2    5000252449    0.000100                    3.500000      3.199000   
3    5000252450    0.000900                    3.500000      3.199000   
4    5000252524    0.000400                    3.500000      3.199000   
..          ...         ...                         ...           ...   
219  5001342228    0.001000                    3.500000      3.199000   
220  5001342229    0.000300                    3.500000      3.199000   
221  5002343705    0.000300                    3.500000      3.199000   
222  5002343706    0.000100                    3.500000      3.199000   
223  5002343707    0.000200                    3.500000      3.199000   

      Volume       cr  
0   6.317600 0.000000  
1   6.317600 0.000000  
2   6.302500 0.000000  
3   9.806700